# Saccade‑aligned LFP average pipeline

**Environment:** Use the `eye_repo` conda environment.

This pipeline loads the exported `all_saccade_collection`, infers the required (animal, block) set, instantiates only those BlockSync blocks, filters to **synchronized (binocular) saccades only**, and computes **averaged LFP traces** aligned to saccade onset (t = 0), with **one average trace per animal** (no mixing across animals).

**Prerequisites:**
- Run `saccade_collection_pipeline.ipynb` and export `all_saccade_collection` to the configured `export_dir`.
- Electrophysiology (OE) data available for each block (`oe_rec` on BlockSync).

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eye_tracking_system_tools.preprocessing import BlockSync
from eye_tracking_system_tools.preprocessing import utility_functions as uf

## Config

- `experiment_path`: parent of animal folders (same as saccade pipeline).
- `collection_path`: path to exported `all_saccade_collection.csv` (or use `export_dir` / `export_filename` from saccade pipeline).
- `lfp_channels`: list of OE channel numbers for LFP (e.g. `[1]`).
- `window_ms`: total window around saccade onset (ms). We use **± half_window**, so 100 ms → -50 to +50 ms.

In [ ]:
experiment_path = Path(r"D:\sample_data_for_eye_repo")
export_dir = experiment_path / "analysis" / "saccade_collections"
export_filename = "all_saccade_collection.csv"
collection_path = export_dir / export_filename

lfp_channels = [1]   # OE channel numbers for LFP
window_ms = 200.0    # total window around onset (ms); we use ± window_ms/2
half_window_ms = window_ms / 2

## Load `all_saccade_collection` and infer required (animal, block) set

Load the exported collection, then derive unique `(animal, block)` pairs to instantiate only those BlockSync blocks.

In [ ]:
all_saccade_collection = pd.read_csv(collection_path)
required = ["animal", "block", "Main", "Sub", "saccade_on_ms"]
missing = [c for c in required if c not in all_saccade_collection.columns]
if missing:
    raise ValueError(f"all_saccade_collection missing columns: {missing}")
print(f"Loaded {len(all_saccade_collection):,} rows from {collection_path.name}")

ab = all_saccade_collection[["animal", "block"]].drop_duplicates()
animals = ab["animal"].dropna().unique().astype(str).tolist()
block_lists = [
    [int(b) for b in ab.loc[ab["animal"] == a, "block"].dropna().unique().astype(str).tolist()]
    for a in animals
]
print(f"Inferred animals: {animals}; block_lists: {block_lists}")

## Build `block_collection` and `block_dict` for inferred (animal, block) only

Reuse the same helpers as the saccade pipeline. We only need BlockSync instances with `oe_rec`; no need to load `final_sync_df` or eye data for LFP extraction.

In [ ]:
def create_block_collections(animals, block_lists, experiment_path, bad_blocks=None):
    """Build block_collection and block_dict from animals and block lists."""
    if bad_blocks is None:
        bad_blocks = []
    block_collection = []
    block_dict = {}
    for animal, blocks in zip(animals, block_lists):
        current = uf.block_generator(
            block_numbers=blocks,
            experiment_path=experiment_path,
            animal=animal,
            bad_blocks=bad_blocks,

        )
        block_collection.extend(current)
        for b in current:
            block_dict[f"{animal}_block_{b.block_num}"] = b
    return block_collection, block_dict

block_collection, block_dict = create_block_collections(
    animals=animals,
    block_lists=block_lists,
    experiment_path=experiment_path,
    bad_blocks=[],

)
print(f"Blocks: {list(block_dict.keys())}")

## Filter saccades into sub-categories (with / without head movements)

Define sub-categories:
1. **Synchronized** – both eyes move at same time (one row per pair, no double-count).
2. **Monocular left** – only left eye crossed threshold (unpaired).
3. **Monocular right** – only right eye crossed threshold (unpaired).
4. **Conjugated** – synchronized saccades with similar direction (L–R angle difference ≤ 45°).
5. **Non-conjugated** – synchronized saccades with divergent direction (L–R angle difference > 45°).

For each category we keep two versions: **with head movements** (include all) and **no head movements** (exclude rows where `head_movement` is True). The `head_movement` column is set by the saccade collection pipeline via `block.block_get_lizard_movement()`: True if any movement epoch (`t_mov_ms` in `block.liz_mov_df`) falls inside the saccade interval `[saccade_on_ms, saccade_off_ms]`, else False. If `head_movement` is missing from the collection (e.g. pipeline run before labeling or no `lizMov.mat`), both versions use all rows and a note is printed.

In [ ]:
# Head-movement mask: exclude saccades during head movement for "no_head" version
if "head_movement" in all_saccade_collection.columns:
    mask_no_head = ~all_saccade_collection["head_movement"].fillna(False).astype(bool)
    mask_with_head = np.ones(len(all_saccade_collection), dtype=bool)  # include all
else:
    mask_no_head = np.ones(len(all_saccade_collection), dtype=bool)
    mask_with_head = np.ones(len(all_saccade_collection), dtype=bool)
    print("Note: 'head_movement' column not in collection; with_head and no_head use same rows.")

# 1) Synchronized (one row per pair: keep Sub == "L" only)
synced = all_saccade_collection.dropna(subset=["Main"]).copy()
synced_one_per_pair = synced.query('Sub == "L"').copy()
synced_with_head = synced_one_per_pair.copy()
synced_no_head = synced_one_per_pair.loc[synced_one_per_pair.index.intersection(all_saccade_collection.index[mask_no_head])].copy()
print(f"1. Synced (one per pair): with_head={len(synced_with_head)}, no_head={len(synced_no_head)}")

# 2) Monocular left (Main is NaN, eye == L)
mono_left = all_saccade_collection[all_saccade_collection["Main"].isna() & (all_saccade_collection["eye"] == "L")].copy()
mono_left_with_head = mono_left.copy()
mono_left_no_head = mono_left.loc[mono_left.index.intersection(all_saccade_collection.index[mask_no_head])].copy()
print(f"2. Monocular left: with_head={len(mono_left_with_head)}, no_head={len(mono_left_no_head)}")

# 3) Monocular right (Main is NaN, eye == R)
mono_right = all_saccade_collection[all_saccade_collection["Main"].isna() & (all_saccade_collection["eye"] == "R")].copy()
mono_right_with_head = mono_right.copy()
mono_right_no_head = mono_right.loc[mono_right.index.intersection(all_saccade_collection.index[mask_no_head])].copy()
print(f"3. Monocular right: with_head={len(mono_right_with_head)}, no_head={len(mono_right_no_head)}")

# 4) Conjugated / 5) Non-conjugated: need L–R angle difference per synced pair (requires "angle" column)
CONJUGATED_DEG = 45  # within this many degrees = conjugated
synced_l_full = synced.query('Sub == "L"').copy()
if "angle" in synced.columns:
    synced_l = synced.query('Sub == "L"')[["Main", "animal", "block", "angle", "saccade_on_ms"]].rename(columns={"angle": "angle_L"})
    synced_r = synced.query('Sub == "R"')[["Main", "angle"]].rename(columns={"angle": "angle_R"})
    pair_angles = synced_l.merge(synced_r, on="Main", how="inner")
    pair_angles["angle_diff"] = np.abs(pair_angles["angle_L"] - pair_angles["angle_R"])
    pair_angles["angle_diff"] = np.minimum(pair_angles["angle_diff"], 360 - pair_angles["angle_diff"])
    pair_angles["conjugated"] = pair_angles["angle_diff"] <= CONJUGATED_DEG
    pair_angles = pair_angles.set_index("Main")
    conj_main_ids = pair_angles.index[pair_angles["conjugated"]].values
    non_conj_main_ids = pair_angles.index[~pair_angles["conjugated"]].values
    conjugated_rows = synced_l_full[synced_l_full["Main"].isin(conj_main_ids)].copy()
    non_conjugated_rows = synced_l_full[synced_l_full["Main"].isin(non_conj_main_ids)].copy()
else:
    conjugated_rows = pd.DataFrame()
    non_conjugated_rows = pd.DataFrame()
    print("Note: 'angle' column missing; conjugated and non-conjugated collections are empty.")

conjugated_with_head = conjugated_rows.copy()
conjugated_no_head = conjugated_rows.loc[conjugated_rows.index.intersection(all_saccade_collection.index[mask_no_head])].copy()
non_conjugated_with_head = non_conjugated_rows.copy()
non_conjugated_no_head = non_conjugated_rows.loc[non_conjugated_rows.index.intersection(all_saccade_collection.index[mask_no_head])].copy()
print(f"4. Conjugated (angle diff ≤{CONJUGATED_DEG}°): with_head={len(conjugated_with_head)}, no_head={len(conjugated_no_head)}")
print(f"5. Non-conjugated (angle diff >{CONJUGATED_DEG}°): with_head={len(non_conjugated_with_head)}, no_head={len(non_conjugated_no_head)}")

# Collect all filtered DataFrames for iteration (name, df) for LFP and plotting
filtered_collections = [
    ("Synced (with head)", synced_with_head),
    ("Synced (no head)", synced_no_head),
    ("Monocular left (with head)", mono_left_with_head),
    ("Monocular left (no head)", mono_left_no_head),
    ("Monocular right (with head)", mono_right_with_head),
    ("Monocular right (no head)", mono_right_no_head),
    ("Conjugated (with head)", conjugated_with_head),
    ("Conjugated (no head)", conjugated_no_head),
    ("Non-conjugated (with head)", non_conjugated_with_head),
    ("Non-conjugated (no head)", non_conjugated_no_head),
]

## Extract LFP windows and average per animal (per filtered collection)

For each saccade in a filtered collection we fetch `window_ms` of LFP centered on saccade onset (onset → 0). We use **`saccade_on_ms`** from the collection (same timebase as block sync). `start = saccade_on_ms - half_window_ms`. We skip windows outside `[0, recordingDuration_ms)`. We **do not mix** traces across animals: one average trace per animal per filtered collection.

*Time alignment:* `saccade_on_ms` is assumed to match the OE recording timebase. If LFP traces look misaligned, verify sync pipeline output (e.g. `ms_axis` vs OE) and adjust.

In [ ]:
all_saccade_collection.head()

In [ ]:
window_ms = 600
half_window_ms = window_ms / 2

def _block_key(animal, block):
    b = str(block).strip()
    if len(b) < 3:
        b = b.zfill(3)
    return f"{animal}_block_{b}"

def extract_mean_lfp_per_animal(collection_df, block_dict, lfp_channels, window_ms, half_window_ms, animals):
    """Extract LFP windows for each saccade in collection_df and average per animal. Returns dict animal -> (t_rel_ms, mean_lfp)."""
    mean_lfp_per_animal = {}
    for animal in animals:
        rows = collection_df[collection_df["animal"] == animal]
        if rows.empty:
            continue
        windows = []
        for _, r in rows.iterrows():
            key = _block_key(r["animal"], r["block"])
            block = block_dict.get(key)
            if block is None or not hasattr(block, "oe_rec") or block.oe_rec is None:
                continue
            onset_ms = float(r["saccade_on_ms"])
            start_ms = onset_ms - half_window_ms
            dur_ms = getattr(block.oe_rec, "recordingDuration_ms", None)
            if dur_ms is not None:
                dur_ms = float(dur_ms) if not hasattr(dur_ms, "item") else float(dur_ms.item())
            if dur_ms is not None and (start_ms < 0 or start_ms + window_ms > dur_ms):
                continue
            try:
                start_arr = np.atleast_2d(np.asarray([start_ms], dtype=float))
                data, ts = block.oe_rec.get_data(
                    lfp_channels,
                    start_arr,
                    window_ms,
                    convert_microvolts=True,
                    return_timestamps=True,
                    repress_output=True,
                )
            except Exception:
                continue
            if data is None or data.size == 0:
                continue
            win = data[0, 0, :]
            windows.append(win)
        if not windows:
            continue
        stack = np.stack(windows)
        mean_lfp = np.nanmean(stack, axis=0)
        n_samp = mean_lfp.size
        t_rel_ms = np.linspace(-half_window_ms, half_window_ms, n_samp, endpoint=True)
        mean_lfp_per_animal[animal] = (t_rel_ms, mean_lfp)
    return mean_lfp_per_animal

# Compute mean LFP per animal for each filtered collection
mean_lfp_by_filter = {}
for name, df in filtered_collections:
    if df.empty:
        mean_lfp_by_filter[name] = {}
        continue
    mean_lfp_by_filter[name] = extract_mean_lfp_per_animal(
        df, block_dict, lfp_channels, window_ms, half_window_ms, animals
    )
    n_windows = sum(len(df[df["animal"] == a]) for a in mean_lfp_by_filter[name])
    print(f"{name}: n_saccades={len(df)}, animals with LFP={len(mean_lfp_by_filter[name])}")

## Plot averaged LFP per filtered collection

One figure per filtered collection. Time axis: ms relative to saccade onset. One mean trace per animal per plot; **no mixing** across animals.

In [ ]:
for name, df in filtered_collections:
    mean_lfp_per_animal = mean_lfp_by_filter[name]
    if not mean_lfp_per_animal:
        continue
    fig, ax = plt.subplots(figsize=(8, 4))
    for animal, (t_rel, lfp) in mean_lfp_per_animal.items():
        n_events = len(df[df["animal"] == animal])
        ax.plot(t_rel, lfp, label=f"{animal} (n={n_events})")
    ax.axvline(0, color="k", ls="--", alpha=0.5)
    ax.set_xlabel("Time relative to saccade onset (ms)")
    ax.set_ylabel("LFP (microV)")
    ax.set_title(f"Averaged LFP aligned to saccade onset: {name}")
    ax.legend()
    ax.set_xlim(-half_window_ms, half_window_ms)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

## Custom LFP average export (single animal, selected electrodes and conditions)

Choose **one animal**, a **list of electrode channels**, and a **list of (name, dataframe) pairs** for differently filtered saccade collections. The next cells will produce a **high-quality PDF** with one averaged LFP plot per (condition × electrode). For example: 2 conditions and 2 electrodes → 4 plots.

In [ ]:
# --- User choices for custom LFP export ---
# Pick one animal (must be in the pipeline's inferred animals)
selected_animal = "PV_126"  # e.g. animals[0]

# OE channel numbers for LFP (one row per electrode in the output figure)
electrodes = [1, 15]

# List of (label, dataframe) for each saccade condition (use your pre-filtered dfs)
# Example: [("condition_A", df1), ("condition_B", df2)] — each becomes a column in the PDF
saccade_collections_custom = [
    ("Synced (with head)", synced_with_head),
    ("Synced (no head)", synced_no_head),
]

# Where to save the PDF: set a path here, or leave None to auto-generate a unique name (animal + blocks)
pdf_save_path = None  # e.g. Path(export_dir) / "my_custom.pdf"
if pdf_save_path is None:
    _animal_blocks = block_lists[animals.index(selected_animal)]
    _blocks_str = "_".join(map(str, sorted(_animal_blocks)))
    _base = f"lfp_average_{selected_animal}_blocks_{_blocks_str}.pdf"
    pdf_save_path = (export_dir / _base) if export_dir.exists() else Path(_base)
if pdf_save_path:
    print(f"PDF will be saved to: {pdf_save_path}")

In [ ]:
# Build mean LFP per (condition, electrode) for the selected animal only, then save to PDF
from matplotlib.backends.backend_pdf import PdfPages

if not pdf_save_path:
    raise ValueError("Set pdf_save_path or choose a file in the previous cell.")

# If pdf_save_path is a directory, write the auto-named file inside it (avoids PermissionError)
if pdf_save_path.is_dir():
    _animal_blocks = block_lists[animals.index(selected_animal)]
    _blocks_str = "_".join(map(str, sorted(_animal_blocks)))
    _fname = f"lfp_average_{selected_animal}_blocks_{_blocks_str}.pdf"
    pdf_save_path = pdf_save_path / _fname
    print(f"Using path: {pdf_save_path}")

# Filter each collection to selected animal and compute mean LFP per electrode
n_conditions = len(saccade_collections_custom)
n_electrodes = len(electrodes)
# (cond_name, ch) -> (t_rel_ms, mean_lfp, n_events)
results = {}
for cond_name, df in saccade_collections_custom:
    df_animal = df[df["animal"] == selected_animal]
    if df_animal.empty:
        print(f"Warning: no rows for animal {selected_animal} in condition '{cond_name}'; skipping.")
        continue
    for ch in electrodes:
        mean_lfp_dict = extract_mean_lfp_per_animal(
            df_animal, block_dict, [ch], window_ms, half_window_ms, [selected_animal]
        )
        if selected_animal not in mean_lfp_dict:
            print(f"Warning: no LFP for {selected_animal} ch {ch} in '{cond_name}'; skipping.")
            continue
        t_rel, mean_lfp = mean_lfp_dict[selected_animal]
        n_events = len(df_animal)
        results[(cond_name, ch)] = (t_rel, mean_lfp, n_events)

if not results:
    raise ValueError("No LFP data for the selected animal/electrodes/conditions. Check filters and block_dict.")

# Layout: rows = electrodes, columns = filtered conditions (no overwrite from generic names)
fig, axes = plt.subplots(
    n_electrodes, n_conditions,
    figsize=(4 * n_conditions, 3.5 * n_electrodes),
    squeeze=False,
)
for row, ch in enumerate(electrodes):
    for col, (cond_name, _) in enumerate(saccade_collections_custom):
        ax = axes[row, col]
        key = (cond_name, ch)
        if key in results:
            t_rel, mean_lfp, n_events = results[key]
            ax.plot(t_rel, mean_lfp, color="C0")
            ax.set_title(f"{cond_name}\nCh {ch} (n={n_events})")
        else:
            ax.set_title(f"{cond_name}\nCh {ch} (no data)")
        ax.axvline(0, color="k", ls="--", alpha=0.5)
        ax.set_xlabel("Time rel. saccade onset (ms)")
        ax.set_ylabel("LFP (µV)")
        ax.set_xlim(-half_window_ms, half_window_ms)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
# PDF metadata: internal name with animal and blocks for identification
_animal_blocks = block_lists[animals.index(selected_animal)]
_pdf_title = f"LFP average — {selected_animal} blocks {_animal_blocks}"
with PdfPages(pdf_save_path) as pdf:
    _meta = pdf.infodict()
    _meta["Title"] = _pdf_title
    pdf.savefig(fig, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved {len(results)} LFP average plot(s) to {pdf_save_path}")